# Party competition: stipulated synthetic compatibility computation

Party shares, membership totals, and approval are stipulated. Nothing below detects strategic interaction, a political effect, a cycle, a regime, or a motion operator.

The legacy compatibility pipeline is scientifically flawed as a detector, has no validated detection capability, and is not a mixture-motion estimator. A centered-log-ratio motion-estimator repair is designed but is not implemented in this notebook. The computations below expose compatibility behavior only.

NMF is a full-sample reconstruction tool here. The same nonnegative observation matrix is passed to NMF and membership re-inference. Every R2 is in-sample. Equilibrium records describe only the fitted matrix; they are not findings about the data-generating process.


In [ ]:
# Imports and deterministic seed
import os
import random
import numpy as np
import ts2eg as gm

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
TS2EG_CI = os.getenv("TS2EG_CI", "0") == "1"


## Stipulated party shares and membership counts


In [ ]:
rng = np.random.default_rng(SEED)
T = 72
t = np.arange(T)
base = np.array([0.36, 0.29, 0.21, 0.14])[:, None]
variation = np.vstack([
    0.05 * np.sin(2 * np.pi * t / 24),
    0.04 * np.cos(2 * np.pi * (t + 4) / 30),
    0.03 * np.sin(2 * np.pi * (t + 7) / 20),
    0.02 * np.cos(2 * np.pi * (t + 2) / 18),
])
X_model = np.maximum(base + variation + 0.008 * rng.standard_normal((4, T)), 1e-8)
X_model /= X_model.sum(axis=0, keepdims=True)
membership_total = 500_000 * np.exp(0.04 * rng.standard_normal(T))
counts = X_model * membership_total
print("party shares:", X_model.shape, "membership counts:", counts.shape)


## Growth payoff compatibility computation


In [ ]:
v_growth = gm.growth_payoffs(counts, dt=1.0, pad="edge")
S, H = gm.nmf_on_X(X_model, k=3, iters=120, seed=SEED, normalize="l2")
fit_growth = gm.estimate_A_from_series(S, X_model, v_growth, k=3, ridge=1e-2)
records = gm.find_ESS(fit_growth["A"], max_support=3)
summary = [
    (r["support"], r["ess_status"], r["stability_status"])
    for r in records if r["is_nash"]
]
print("Legacy descriptive growth payoff-field R2 (in-sample):", round(fit_growth["R2"], 3))
print("Fitted-matrix Nash/ESS/stability descriptions:", summary)


## Stipulated same-time approval association

Rolling LOFO values are in-sample predictive associations only; they are neither future vote forecasts nor causal effects.


In [ ]:
approval_score = (
    0.5 * X_model[0] - 0.3 * X_model[1] + 0.2 * X_model[2]
    + 0.04 * rng.standard_normal(T)
)
v_lofo = gm.info_gain_payoffs(X_model, approval_score, ridge=1e-2, window=18)
fit_lofo = gm.estimate_A_from_series(S, X_model, v_lofo, k=3, ridge=1e-2)
print("Legacy descriptive LOFO payoff-field R2 (in-sample):", round(fit_lofo["R2"], 3))
